# Module 6: Cedar Policies -- Deterministic Guardrails

![Overview](../shared/img/06.drawio.png)

In this module, you will add **Cedar policies** to Aria's Gateway using the AgentCore Policy service. Unlike prompt-based guardrails that rely on the LLM to comply, Cedar policies are **deterministic** -- they evaluate at the Gateway boundary, outside agent code, with 100% reliability.

There is no new agent code for this module. Aria continues to run V4 from Module 5. The focus here is entirely on the policy layer.

## What you will learn

- **AgentCore Policy**: How to create a Policy Engine and attach it to a Gateway
- **Cedar language**: The `permit(principal, action, resource) when { conditions }` syntax
- **Deterministic guardrails**: Why policies are more reliable than prompt-based rules
- **Business rules**: Enforcing that tasks cannot be created with status "completed"
- **ENFORCE vs LOG_ONLY**: Choosing the right enforcement mode

## Catch-up

The following cell ensures all prerequisites from earlier modules are in place. If you completed Modules 0-5, this will verify everything. If you skipped ahead, it will create the missing resources automatically.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("06")

## Why deterministic policies?


Consider two approaches to enforcing the rule *"users cannot create tasks that are already marked completed"*:


| Approach | How it works | Reliability |
|---|---|---|
| **Prompt-based** | Add to system prompt: "Never create a task with status completed" | Probabilistic -- the LLM *usually* follows instructions, but can be jailbroken or simply forget |
| **Cedar policy** | `permit(...) unless { action is create_task and status is completed }` | Deterministic -- the Gateway evaluates the policy *before* the request reaches the API. 100% reliable. |


Cedar policies evaluate at the **Gateway boundary**, completely outside the agent's code and the LLM's reasoning loop. The agent never even sees a denied request -- the Gateway blocks it before it reaches the target API.


This is the same Cedar language used by [Amazon Verified Permissions](https://aws.amazon.com/verified-permissions/) and is purpose-built for authorization decisions.


> **Docs**: [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Understanding Cedar

Cedar policies follow a **principal / action / resource** model with conditions:

```cedar
permit(
  principal,                                                // Who (any authenticated user)
  action == AgentCore::Action::"TaskApi___create_task",    // What (create task)
  resource == AgentCore::Gateway::"<gateway-arn>"           // Where (this Gateway)
) when {
  !(context.input has status && context.input.status == "completed")
};
```

### Key concepts

- **Default deny** -- Cedar denies any request that is not explicitly permitted. Each tool action needs its own `permit` policy.
- **Actions** are named using the pattern `<target-name>___<tool-name>`, matching the tool names defined in your Gateway target's `toolOverrides`. For our task API: `TaskApi___list_tasks`, `TaskApi___create_task`, `TaskApi___update_task`, `TaskApi___delete_task`
- **Resource** is the Gateway ARN: `AgentCore::Gateway::"<ARN>"`
- **Conditions** use `context.input` to inspect the request body fields. Every policy requires a `when` condition that references `context.input` -- the Policy Engine rejects unconditional permits as overly broad.
- **Sentinel conditions** -- For policies that should allow all requests, use a condition that is always true: `!(context.input has field && context.input.field == "__blocked__")`. This satisfies the validator while never actually blocking anything.

### The business rule

Tasks cannot be CREATED with status "completed". They must start as "pending" or "in_progress" and transition through an update. This enforces a proper task lifecycle.

> **Go deeper:** [Cedar policy language](https://www.cedarpolicy.com/) | [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Setup

First we connect to the AgentCore control plane and load the Gateway configuration from Module 5.

In [ ]:
import boto3, time
from botocore.exceptions import ClientError
import sys; sys.path.insert(0, '..')
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)

gw_config = utils.get_gateway_config(control)
gateway_id = gw_config["gateway_id"]
gateway_arn = gw_config["gateway_arn"]

print(f"Gateway ID:  {gateway_id}")
print(f"Gateway ARN: {gateway_arn}")

## Create the Policy Engine

A Policy Engine is the container that holds Cedar policies and evaluates them against incoming requests. We create one and wait for it to become ACTIVE.

In [ ]:
# Step 1: Create the Policy Engine
try:
    resp = control.create_policy_engine(
        name="aria_policy_engine",
        description="Policy engine for Aria -- Cedar policy enforcement on Gateway tools",
    )
    engine_id = resp["policyEngineId"]
    engine_arn = resp["policyEngineArn"]
    print(f"Policy Engine created: {engine_id}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Policy Engine already exists -- looking it up...")
        paginator = control.get_paginator("list_policy_engines")
        for page in paginator.paginate():
            for eng in page.get("policyEngines", page.get("items", [])):
                if eng["name"] == "aria_policy_engine":
                    engine_id = eng["policyEngineId"]
                    detail = control.get_policy_engine(policyEngineId=engine_id)
                    engine_arn = detail["policyEngineArn"]
                    print(f"Found existing: {engine_id}")
                    break
    else:
        raise

print(f"Engine ID:  {engine_id}")
print(f"Engine ARN: {engine_arn}")

In [ ]:
# Step 1b: Wait for ACTIVE status
result = utils.poll_until(
    describe_fn=lambda: control.get_policy_engine(policyEngineId=engine_id),
    status_path="status",
    target_statuses={"ACTIVE", "READY"},
    label="Policy Engine",
    interval=10,
    timeout=300,
)
print(f"\nPolicy Engine is ACTIVE: {engine_id}")

## Create Cedar policies

Cedar's **default deny** means every tool action needs an explicit permit. We need four policies -- one per Gateway tool:

| Policy | Action | Condition |
|---|---|---|
| `permit_list_tasks` | `TaskApi___list_tasks` | Block when `status == "completed"` |
| `permit_create_task` | `TaskApi___create_task` | Block when `status == "completed"` |
| `permit_update_task` | `TaskApi___update_task` | Allow all (sentinel condition) |
| `permit_delete_task` | `TaskApi___delete_task` | Allow all (sentinel condition) |

The `create_task` and `list_tasks` policies enforce the business rule: tasks cannot be created (or filtered) with status "completed". The `update_task` and `delete_task` policies permit all operations.

> **Important:** Every policy requires a `when` condition that references `context.input`. The Policy Engine rejects unconditional permits as overly broad. For policies that should allow all requests, use a sentinel condition that is always true (e.g., `!(context.input has field && context.input.field == "__blocked__")`).

In [ ]:
# Define all four Cedar policies
# Note: Every policy needs a 'when' condition referencing context.input --
# unconditional permits are rejected by the Policy Engine as overly broad.
policies = [
    {
        "name": "permit_list_tasks",
        "description": "Permit listing tasks. Blocks listing only completed tasks.",
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___list_tasks",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "completed")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_create_task",
        "description": "Permit creating tasks, but NOT with status completed.",
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___create_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "completed")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_update_task",
        "description": "Permit all task updates including setting status to completed.",
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___update_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has status && context.input.status == "__blocked__")\n'
            f'}};\n'
        ),
    },
    {
        "name": "permit_delete_task",
        "description": "Permit deleting tasks.",
        "cedar": (
            f'permit(\n'
            f'  principal,\n'
            f'  action == AgentCore::Action::"TaskApi___delete_task",\n'
            f'  resource == AgentCore::Gateway::"{gateway_arn}"\n'
            f') when {{\n'
            f'  !(context.input has id && context.input.id == "__blocked__")\n'
            f'}};\n'
        ),
    },
]

# Create each policy (idempotent -- skips if already exists)
for p in policies:
    try:
        resp = control.create_policy(
            policyEngineId=engine_id,
            name=p["name"],
            description=p["description"],
            definition={"cedar": {"statement": p["cedar"]}},
        )
        print(f"Created: {p['name']} (ID: {resp['policyId']})")
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
            print(f"Already exists: {p['name']}")
        else:
            raise

print("\nBusiness rule Cedar (create_task):")
for line in policies[1]["cedar"].strip().splitlines():
    print(f"  {line}")

### NL2Cedar: generating policies from natural language

AgentCore Policy also supports **NL2Cedar** -- an API that translates plain-English descriptions into Cedar policies. Instead of writing Cedar by hand, you describe the intent:

```python
# Start policy generation from natural language
control.start_policy_generation(
    policyEngineId=engine_id,
    name="gen_refund_limit",
    resource={"arn": gateway_arn},
    content={"rawText": "Allow customer service agents to process refunds up to 500 dollars"},
)
```

NL2Cedar analyzes the Gateway's tools and input schemas to produce valid Cedar with the right action names and `context.input` conditions. This is particularly useful when policies have complex conditions that are easier to express in natural language than in Cedar syntax.

> **Go deeper:** [NL2Cedar policy generation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## Attach Policy Engine to Gateway

Now we attach the policy engine to the Gateway in **ENFORCE** mode. This means any request that violates a policy will be **denied** -- the agent will receive an error instead of the tool result.

In [ ]:
import time
time.sleep(5)  # Allow policies to propagate

# Step 3: Attach policy engine to Gateway in ENFORCE mode
gw = control.get_gateway(gatewayIdentifier=gateway_id)

update_params = {
    "gatewayIdentifier": gateway_id,
    "name": gw["name"],
    "roleArn": gw["roleArn"],
    "protocolType": gw["protocolType"],
    "authorizerType": gw["authorizerType"],
    "policyEngineConfiguration": {
        "arn": engine_arn,
        "mode": "ENFORCE",
    },
}
if "authorizerConfiguration" in gw:
    update_params["authorizerConfiguration"] = gw["authorizerConfiguration"]

control.update_gateway(**update_params)
print("Policy engine attached to Gateway in ENFORCE mode")
print(f"  Engine: {engine_id}")
print(f"  Mode:   ENFORCE")

### Wait for Gateway to update

In [ ]:
result = utils.poll_until(
    describe_fn=lambda: control.get_gateway(gatewayIdentifier=gateway_id),
    status_path="status",
    target_statuses={"ACTIVE", "READY", "UPDATE_COMPLETE"},
    label="Gateway",
    interval=10,
    timeout=300,
)
print(f"\nGateway updated with policy engine")

### Save configuration

In [ ]:
# Save policy config for later modules
policy_config = {
    "policy_engine_id": engine_id,
    "policy_engine_arn": engine_arn,
    "enforcement_mode": "ENFORCE",
    "gateway_id": gateway_id,
}
utils.save_config("policy", policy_config)
print(f"Configuration saved")
print(f"  Engine ID: {engine_id}")
print(f"  Mode:      ENFORCE")

## Test Cedar policy enforcement

Let's test the policies. We use the shared `test_agent` helper with a JWT token. Each invocation flows through:

```
test_agent.invoke() --> Runtime --> Agent --> Gateway --> Cedar Policy --> Target API
```

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

### Test 1: Create a task with status "completed" -- should be DENIED

This request explicitly asks to create a task with status "completed". The `permit_create_task` policy blocks this because the condition `!(context.input has status && context.input.status == "completed")` evaluates to false.

In [ ]:
# This should be DENIED by the Cedar policy
result = test_agent.invoke(
    "Create a task with status completed: Test policy bypass",
    jwt_token=jwt_token,
)

### Test 2: Create a task without specifying status -- should SUCCEED

When no status is specified, the API defaults to "pending". The `permit_create_task` policy allows this because the condition `!(context.input has status && context.input.status == "completed")` evaluates to true (no status field present).

In [ ]:
# This should SUCCEED -- no status means default to "pending"
result = test_agent.invoke(
    "Create a task: Test policy enforcement",
    jwt_token=jwt_token,
)

### Test 3: Update a task status to completed -- should SUCCEED

The `permit_update_task` policy permits all updates unconditionally, including setting status to "completed". The business rule only restricts *creation*, not updates.

In [ ]:
# This should SUCCEED -- updates to "completed" are allowed
result = test_agent.invoke(
    "Update the task status to completed",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)

## ENFORCE vs LOG_ONLY

| Mode | Behavior | Use case |
|---|---|---|
| **ENFORCE** | Denies requests that violate policies. The agent receives an error. | Production — hard enforcement of business rules |
| **LOG_ONLY** | Logs policy decisions but allows all requests through. | Testing — verify policies work before enforcing |

In LOG_ONLY mode, the "create task with status completed" request above would succeed, but the policy decision would be logged for review. This is useful when rolling out new policies — start with LOG_ONLY to monitor behavior, then switch to ENFORCE once you are confident.

### Try it interactively

Test the policy enforcement through the CLI chat. Try asking Aria to create a task that is already completed — the policy should block it:

```bash
cd /workshop/06-policy
python ../shared/chat.py --auth
```

Try these prompts:
1. `Create a task called "finish report" with status completed` — should be denied
2. `Create a task called "finish report"` — should succeed (defaults to pending)
3. `Mark the finish report task as completed` — should succeed (updates are allowed)

> **Go deeper:** [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html) | [NL2Cedar policy generation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)

## What's next

Cedar policies now govern Aria's behavior **deterministically** at the Gateway boundary. No matter what prompt injection or jailbreak is attempted, the policy engine will block requests that violate the business rules.

In **Module 7**, we add **observability and evaluations** -- OpenTelemetry tracing for monitoring and custom evaluators that score the agent's performance in production.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("06")

---

**Next up: [Module 7 -- Observability and Evaluations](../07-observability-evaluations/notebook.ipynb)**